# Module 8: Extended Thinking Walkthrough

Extended thinking gives Claude a private scratchpad to reason through complex problems before answering.
This notebook shows how to enable it, read thinking blocks, and decide when it's worth the cost.


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from anthropic import Anthropic
from dotenv import load_dotenv
load_dotenv()

client = Anthropic()
MODEL  = 'claude-opus-4-7'   # Extended thinking requires Opus
print('Client ready — using', MODEL)

## 1. Enable Extended Thinking

Add `thinking={'type': 'enabled', 'budget_tokens': N}`. The budget must be ≥ 1024.

In [ ]:
def ask_with_thinking(prompt: str, budget: int = 4000) -> dict:
    response = client.messages.create(
        model=MODEL,
        max_tokens=budget + 2048,   # must cover budget + answer
        thinking={'type': 'enabled', 'budget_tokens': budget},
        messages=[{'role': 'user', 'content': prompt}]
    )
    result = {'thinking': '', 'answer': '', 'usage': response.usage}
    for block in response.content:
        if block.type == 'thinking': result['thinking'] = block.thinking
        elif block.type == 'text':   result['answer']   = block.text
    return result

print('Function defined.')

## 2. Simple Question — Does Thinking Help?

For easy questions, Claude barely uses its thinking budget.

In [ ]:
result = ask_with_thinking('What is 2 + 2?', budget=1024)
print(f'Thinking used: {len(result["thinking"])} chars')
print(f'Answer: {result["answer"]}')
print(f'Tokens: {result["usage"].input_tokens} in / {result["usage"].output_tokens} out')

## 3. Hard Multi-Step Problem

Extended thinking shines on problems with multiple constraints.

In [ ]:
hard_problem = """
Three friends share a flat. Alice pays £800/month rent.
Bob pays 1.5× what Charlie pays. Together Bob and Charlie pay £1,200.

1. How much does each person pay?
2. If they split all utilities (£240/month) equally, what is each person's total monthly cost?
3. Alice gets a 10% rent reduction for renewing her lease. What is her new total?
"""

result = ask_with_thinking(hard_problem, budget=4000)

print('=== THINKING SCRATCHPAD (first 800 chars) ===')
print(result['thinking'][:800])
print()
print('=== FINAL ANSWER ===')
print(result['answer'])

## 4. Budget Utilisation

If Claude uses < 50% of the budget, the budget is oversized. Measure and tune it.

In [ ]:
def utilisation(thinking_text: str, budget: int) -> float:
    """Approximate: ~4 chars per token."""
    return len(thinking_text) / (budget * 4)

budgets = [1024, 2000, 4000]
prompt  = 'Is it better to RAG or fine-tune for a customer support chatbot? Consider cost, accuracy, and update frequency.'

print(f'{'Budget':>8}  {'Used (chars)':>14}  {'Utilisation':>12}  {'Verdict':>20}')
print('-' * 60)

for budget in budgets:
    r    = ask_with_thinking(prompt, budget=budget)
    util = utilisation(r['thinking'], budget)
    verdict = 'Good' if 0.4 <= util <= 0.9 else ('Oversized' if util < 0.4 else 'Undersized')
    print(f'{budget:>8}  {len(r["thinking"]):>14}  {util:>11.1%}  {verdict:>20}')

## 5. Your Turn

Try your own complex problem and see how much thinking Claude uses.

In [ ]:
my_problem = """Your problem here."""

result = ask_with_thinking(my_problem, budget=3000)
print('Thinking:', result['thinking'][:500])
print('\nAnswer:', result['answer'])